# Appendix B — GBM Calibration: Known-Answer Synthetic Test

**Fulfills the promise made in the thesis at §5.9.3 / §5.10.1 (p. 82–83):**

> "The GBM calibration code is checked using a known-answer synthetic test: a series with chosen
> parameters is simulated and recalibrated, confirming that volatility recovers reliably while drift
> does not. Full derivation and verification details are provided in the Appendix."

## B.1 Purpose

The EU ETS carbon price is modelled in this thesis as a geometric Brownian motion (GBM), calibrated
from EU ETS Phase 4 auction data (2021–2025) by the method of moments on log-returns (§5.9.3).
Because the calibration window (~4.9 years) is short relative to what is typically needed for reliable
drift estimation in financial time series, this thesis states plainly that GBM drift is the least reliable
calibrated parameter, and that all breakeven-crossing probabilities are drift-sensitive (§5.9.3, p. 81;
§8.4, p. 108).

This appendix tests that claim directly: is the *estimator itself* — not just the real, noisy market data —
responsible for the unreliable drift? The way to answer that is to simulate data with a **known** ground
truth, recalibrate, and see how far the estimate strays from the truth it was built from.

## B.2 The calibration function under test

This is the exact `calibrate_gbm()` function from `CCS_TEA_Model.ipynb` (Cell 37), copied verbatim.
It is a method-of-moments estimator on log-returns: annualised volatility from the standard deviation
of log-returns, and annualised drift from their mean, corrected for the Itô/Jensen term
($+\tfrac{1}{2}\sigma^2$).

In [1]:
import numpy as np

# ── Verbatim from CCS_TEA_Model.ipynb, Cell 37 ─────────────────────────────
def calibrate_gbm(prices, dt=1/252):
    """Calibrate GBM from log-returns. dt=1/252 for business-daily ETS data."""
    prices = np.array(prices, dtype=float)
    prices = prices[(~np.isnan(prices)) & (prices > 0)]
    if len(prices) < 20:
        print("  [WARNING] Too few valid ETS observations for calibration.")
        return np.nan, np.nan
    log_returns = np.diff(np.log(prices))
    log_returns = log_returns[np.isfinite(log_returns)]
    if len(log_returns) < 10:
        return np.nan, np.nan
    sigma_annual = np.std(log_returns) / np.sqrt(dt)
    mu_log_annual = np.mean(log_returns) / dt
    mu_annual = mu_log_annual + 0.5 * sigma_annual**2
    return mu_annual, sigma_annual

## B.3 Ground-truth parameters and sample design

The synthetic test uses the **same parameters this thesis's own calibration produced from the real
EU ETS Phase 4 data**, so the test exercises the estimator under the exact conditions (sample size,
window length, sampling convention) actually used to generate the thesis's reported results —
not an arbitrary or convenient choice of true parameters.

`CCS_TEA_Model.ipynb` (Cell 37), run against `ETS_price_clean_full.csv`, reports:

```
ETS data loaded: 1079 Phase 4 observations (2021-01-29 to 2025-12-15)
ETS calibrated from data: μ=0.306/yr  σ=0.423/yr
```

So the known-answer test below sets **μ_true = 0.306/yr, σ_true = 0.423/yr**, generates synthetic
series of the **same length (n = 1,079 observations)**, and calibrates them with `calibrate_gbm()`
using the **same `dt=1/252` convention** actually used in the notebook (i.e. the calibration function
is exercised exactly as it was for the real data — this is a test of the estimator, not of a different
setup).

In [2]:
# ── Ground truth: this thesis's own real Phase 4 calibration output ────────
mu_true    = 0.306   # /yr — from CCS_TEA_Model.ipynb, Cell 37 (real ETS_price_clean_full.csv)
sigma_true = 0.423   # /yr — from CCS_TEA_Model.ipynb, Cell 37 (real ETS_price_clean_full.csv)
P0         = 100.0
dt         = 1/252   # same convention as calibrate_gbm()'s default, used in the notebook's call
n_obs      = 1079    # same sample size as the real Phase 4 observation count

def simulate_synthetic_series(mu, sigma, n_obs, dt, P0=100.0, seed=None):
    """One synthetic GBM price path with KNOWN true (mu, sigma)."""
    rng = np.random.default_rng(seed)
    log_price = np.log(P0)
    path = [P0]
    for _ in range(n_obs):
        dW = rng.normal(0, np.sqrt(dt))
        log_price += (mu - 0.5 * sigma**2) * dt + sigma * dW
        path.append(np.exp(log_price))
    return path

print(f"Ground truth: mu_true={mu_true}, sigma_true={sigma_true}, n_obs={n_obs}, dt=1/252")

Ground truth: mu_true=0.306, sigma_true=0.423, n_obs=1079, dt=1/252


## B.4 Single-path illustration

The thesis's own wording ("a series with chosen parameters is simulated and recalibrated") describes
a single-path test. This is that single path, using seed 42 for reproducibility (the same seed convention
used elsewhere in this thesis's Monte Carlo, e.g. §5.9.2).

In [3]:
path_single = simulate_synthetic_series(mu_true, sigma_true, n_obs, dt, P0, seed=42)
mu_hat_single, sigma_hat_single = calibrate_gbm(path_single, dt=dt)

print(f"True:      mu={mu_true:.4f}   sigma={sigma_true:.4f}")
print(f"Recovered: mu={mu_hat_single:.4f}   sigma={sigma_hat_single:.4f}")
print(f"\nAbsolute error: "
      f"mu {abs(mu_hat_single-mu_true):.4f} ({abs(mu_hat_single-mu_true)/mu_true*100:.1f}%), "
      f"sigma {abs(sigma_hat_single-sigma_true):.4f} ({abs(sigma_hat_single-sigma_true)/sigma_true*100:.1f}%)")

True:      mu=0.3060   sigma=0.4230
Recovered: mu=0.1288   sigma=0.4163

Absolute error: mu 0.1772 (57.9%), sigma 0.0067 (1.6%)


## B.5 Multi-realization test — is the single path typical?

A single path could, by chance, land on either a lucky or unlucky recovery. To characterise the
*reliability* of the estimator — not just one draw — the same test is repeated across 2,000 independent
synthetic realizations, each of the same length and using the same true parameters.

In [4]:
n_realizations = 2000
rng_master = np.random.default_rng(2026)

mus, sigmas = [], []
for k in range(n_realizations):
    path = simulate_synthetic_series(mu_true, sigma_true, n_obs, dt, P0,
                                      seed=int(rng_master.integers(0, 2**31 - 1)))
    mu_hat, sigma_hat = calibrate_gbm(path, dt=dt)
    mus.append(mu_hat)
    sigmas.append(sigma_hat)

mus = np.array(mus)
sigmas = np.array(sigmas)

print(f"{n_realizations} independent synthetic realizations, n_obs={n_obs} each\n")

print("── Volatility (sigma) recovery ──")
print(f"  Mean recovered:  {sigmas.mean():.4f}  (true: {sigma_true})")
print(f"  Std. dev. across realizations: {sigmas.std():.4f}")
print(f"  Relative bias: {(sigmas.mean()-sigma_true)/sigma_true*100:+.2f}%")
print(f"  Coefficient of variation: {sigmas.std()/sigmas.mean()*100:.2f}%")

print("\n── Drift (mu) recovery ──")
print(f"  Mean recovered:  {mus.mean():.4f}  (true: {mu_true})")
print(f"  Std. dev. across realizations: {mus.std():.4f}")
print(f"  Relative bias: {(mus.mean()-mu_true)/mu_true*100:+.2f}%")
print(f"  Coefficient of variation: {mus.std()/abs(mus.mean())*100:.2f}%")

ratio = (mus.std()/abs(mus.mean())) / (sigmas.std()/sigmas.mean())
print(f"\nRatio of drift CV to volatility CV: {ratio:.1f}x")
print("  -> drift estimates vary this many times more, realization to realization,")
print("     than volatility estimates, for the SAME sample size and true parameters.")

2000 independent synthetic realizations, n_obs=1079 each

── Volatility (sigma) recovery ──
  Mean recovered:  0.4226  (true: 0.423)
  Std. dev. across realizations: 0.0091
  Relative bias: -0.10%
  Coefficient of variation: 2.16%

── Drift (mu) recovery ──
  Mean recovered:  0.3103  (true: 0.306)
  Std. dev. across realizations: 0.2012
  Relative bias: +1.39%
  Coefficient of variation: 64.86%

Ratio of drift CV to volatility CV: 30.0x
  -> drift estimates vary this many times more, realization to realization,
     than volatility estimates, for the SAME sample size and true parameters.


## B.6 Interpretation

Both parameters are recovered **without meaningful bias on average** — the mean recovered volatility
and drift across 2,000 realizations sit within a fraction of a percent of their true values. This confirms
`calibrate_gbm()` is not systematically wrong.

But the two parameters differ sharply in **reliability**: volatility is tightly clustered around the truth
from one realization to the next (coefficient of variation ≈ 2%), while drift is enormously more
variable (coefficient of variation over 30 times larger). The single-path illustration in §B.4 makes this
concrete on one draw: the same series, the same estimator, produced a volatility estimate a couple of
percent off truth and a drift estimate tens of percent off truth.

This confirms, on data with a known ground truth, the statistical property underlying this thesis's own
drift disclosure (§5.9.3, p. 81; §8.4, p. 108): **signal-to-noise for drift estimation scales with the span
of the data in years, not the number of observations**, whereas volatility is estimable from return
dispersion alone and converges much faster. With only a ~4.9-year Phase 4 window, the calibrated ETS
drift used in this thesis (μ = 0.306/yr) should be read as one plausible realization among a wide range
the same estimator could have produced from an equally long alternative history — consistent with this
thesis's treatment of all drift-sensitive outputs (breakeven-crossing probabilities) as illustrative of
revenue uncertainty rather than as point forecasts (§7.4.4, p. 102; §8.4, p. 108).

**Source of record:** `calibrate_gbm()` is copied verbatim from `CCS_TEA_Model.ipynb` (Cell 37), and
the ground-truth parameters (μ=0.306, σ=0.423) are that same cell's own printed output when run
against the real `ETS_price_clean_full.csv`. This validation exercises the same code and the same
calibrated values used to produce the thesis's reported results — not a re-implementation or a
different parameter choice.

## B.7 Independent Re-Derivation from Raw Data

Sections B.1–B.6 above take the ground-truth parameters (μ=0.306/yr, σ=0.423/yr) from
`CCS_TEA_Model.ipynb`'s own printed calibration output. As a further check, this section
re-derives those parameters completely independently, by re-running the same calibration
logic directly against the raw `ETS_price_clean_full.csv` included in this repository
(`data/processed/ETS_price_clean_full.csv`) — rather than trusting the notebook's console
output at face value.

In [5]:
import pandas as pd

EUR_NOK_2025 = 11.50

# ── Replicate load_ets_data() exactly (CCS_TEA_Model.ipynb, Cell 37) ──────
df = pd.read_csv("../data/processed/ETS_price_clean_full.csv", parse_dates=["Date"])
df = df.sort_values("Date")
df = df[df["Date"] >= "2021-01-01"].copy()
print(f"Phase 4 observations: {len(df)} ({df['Date'].min().date()} to {df['Date'].max().date()})")

ets_prices_raw = df["price_eur"].values
mu_check, sigma_check = calibrate_gbm(ets_prices_raw * EUR_NOK_2025)

print(f"\nIndependently re-derived from raw CSV: mu={mu_check:.4f}/yr  sigma={sigma_check:.4f}/yr")
print(f"Ground truth used in this appendix:    mu={mu_true}/yr  sigma={sigma_true}/yr")

assert abs(mu_check - mu_true) < 0.001 and abs(sigma_check - sigma_true) < 0.001, \
    "Independent re-derivation does not match the ground-truth parameters used above."
print("\nPASS: the ground-truth parameters used in this appendix are independently")
print("reproducible from the raw data, not just taken from a console printout.")

Phase 4 observations: 1079 (2021-01-29 to 2025-12-15)

Independently re-derived from raw CSV: mu=0.3062/yr  sigma=0.4226/yr
Ground truth used in this appendix:    mu=0.306/yr  sigma=0.423/yr

PASS: the ground-truth parameters used in this appendix are independently
reproducible from the raw data, not just taken from a console printout.
